# 第3章 智能软件操作系统

上一章我们讲到：**NPU 硬件 → 需要软件驱动 → 需要操作系统管理 → 昇腾全栈软件体系**。本章就来拆开昇腾的软件操作系统，看看嵌入式操作系统到底是什么、昇腾平台又是如何运作的。

> **运行环境**：CANN 9.0.0 / Python 3.11 / Atlas A2 (ARM aarch64) / Ascend 910B3 / 16vCPUs / 32GiB

---

## 1. 什么是嵌入式操作系统

嵌入式操作系统（Embedded Operating System）是运行在嵌入式设备上的操作系统，与桌面/服务器操作系统相比，它有如下特点：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">维度</th>
<th style="text-align: left;">桌面/服务器 OS</th>
<th style="text-align: left;">嵌入式 OS</th>
</tr>
<tr>
<td style="text-align: left;">资源约束</td>
<td style="text-align: left;">资源充足（GB 级内存）</td>
<td style="text-align: left;">资源受限（MB 级内存、有限存储）</td>
</tr>
<tr>
<td style="text-align: left;">实时性</td>
<td style="text-align: left;">无严格实时要求</td>
<td style="text-align: left;">常需硬实时或软实时保证</td>
</tr>
<tr>
<td style="text-align: left;">硬件耦合</td>
<td style="text-align: left;">硬件抽象，通用性强</td>
<td style="text-align: left;">与特定硬件紧密耦合，需定制</td>
</tr>
<tr>
<td style="text-align: left;">典型代表</td>
<td style="text-align: left;">Ubuntu Desktop、Windows</td>
<td style="text-align: left;">嵌入式 Linux、VxWorks、FreeRTOS、openEuler Embedded</td>
</tr>
</table>

嵌入式操作系统是连接底层硬件与上层应用的桥梁，负责**硬件资源管理、进程调度、内存管理、中断处理、文件系统、网络通信**等核心功能。

**表格逐行解读**：
- **资源约束**：桌面/服务器 OS 通常运行在拥有 GB 级内存、数百 GB 存储的机器上，无需过多考虑资源节约；而嵌入式 OS 运行在内存仅有 MB 级、存储有限的 MCU 或 SoC 上，每一字节都需精打细算，因此常使用 BusyBox 等精简工具集。
- **实时性**：桌面 OS 以吞吐量和公平性为目标，不保证某任务在确定时间内完成；嵌入式 OS 常需"硬实时"（错过截止时间即导致系统故障，如汽车安全气囊）或"软实时"（偶尔超时可接受，如视频流），因此常采用 VxWorks、FreeRTOS 等实时内核。
- **硬件耦合**：桌面 OS 通过 HAL（硬件抽象层）屏蔽硬件差异，同一份 Windows 可装在无数种 PC 上；嵌入式 OS 必须为特定 SoC 定制设备树、引脚复用、时钟配置等，与硬件紧密耦合，可移植性较差。
- **典型代表**：桌面领域以 Ubuntu/Windows 为代表；嵌入式领域则有嵌入式 Linux（如 openEuler Embedded）、VxWorks（航天航空）、FreeRTOS（IoT 微控制器）等，各有其擅长的场景。

<img src="../../images/pptx_slide02_002.png" alt="昇腾全栈软件体系" style="display: block; margin-left: 0;" />

---

## 2. 昇腾软件体系架构：分层全栈设计

<img src="./images/pptx_slide04_003.jpg" alt="昇腾软件体系架构" style="display: block; margin-left: 0;" />

昇腾 AI 处理器为人工智能应用提供了强大的算力基础，但硬件能力的释放需要完整的软件体系支撑。昇腾软件体系构建了一个从底层硬件到上层应用的完整技术栈，涵盖了操作系统、驱动、加速库、编程接口、开发工具等多个层次。其设计理念是**软硬件协同与分层解耦**——每一层都构建于下层能力之上，并向上层提供更抽象、更易用的接口与服务，最终目标是将底层专用硬件的强大算力高效、便捷地释放给最终用户和开发者。

### 硬件层：算力的物理承载

昇腾 AI 处理器是整个软件体系运行的基础，包含 **AI Core**（AI 计算核心，基于达芬奇架构）、**AI CPU**（AI 控制CPU，负责算子调度与数据预处理）、**DVPP**（数字视觉预处理模块，负责图像/视频硬件解码与缩放）等计算单元。这些硬件单元提供了算力的物理承载，但需要完整的软件栈才能将其能力释放给上层应用。

### 四层软件架构

昇腾的软件体系自底向上共**四个核心层次**，每一层构建于下层能力之上，向上提供更抽象、更易用的接口：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">层次</th>
<th style="text-align: left;">名称</th>
<th style="text-align: left;">核心职责</th>
</tr>
<tr>
<td style="text-align: left;">第1层</td>
<td style="text-align: left;">操作系统与驱动层</td>
<td style="text-align: left;">硬件资源管理、进程调度、内存管理、中断处理；NPU 设备驱动和固件；昇腾驱动作"翻译官"，将标准 OS 指令转换为硬件信号</td>
</tr>
<tr>
<td style="text-align: left;">第2层</td>
<td style="text-align: left;">异构计算架构层（CANN）</td>
<td style="text-align: left;">连接上层生态与底层硬件的核心桥梁；图编译器、算子库、运行时环境、AscendCL 编程接口；向上支持 PyTorch/TensorFlow/MindSpore，向下通过驱动与 NPU 交互</td>
</tr>
<tr>
<td style="text-align: left;">第3层</td>
<td style="text-align: left;">框架与模型层</td>
<td style="text-align: left;">提供友好开发界面；MindSpore（华为原生）、PyTorch/TensorFlow（通过适配器接入）；ATC 模型转换工具将训练模型转为 .om 离线模型</td>
</tr>
<tr>
<td style="text-align: left;">第4层</td>
<td style="text-align: left;">应用与工具层</td>
<td style="text-align: left;">AI 能力落地到具体场景；行业解决方案（智能安防、图像识别、语音处理）、预训练模型库、MindStudio 集成开发环境</td>
</tr>
</table>

**四层架构逐层解读**：
- **第1层（操作系统与驱动层）**：这是整个软件栈的"地基"。操作系统（以 Linux 为主）负责管理硬件资源，驱动层则为上层提供访问硬件的接口。昇腾驱动作"翻译官"，将标准 Linux 的通用指令转换为昇腾 NPU 能理解的硬件信号。没有这一层，上层软件无法感知硬件的存在。在香橙派上，这一层就是深度定制的 ARM aarch64 Linux 内核及昇腾专用驱动模块。本节后续将重点介绍操作系统的构成、内核、驱动开发等内容。
- **第2层（CANN 异构计算架构层）**：这是连接上层生态与底层硬件的"核心桥梁"，承担着"承上启下"的关键角色。它包含驱动层（管理 NPU 设备）、运行时库（调度计算任务）、任务调度器（Stream 并行管理）和图编译器（将神经网络编译为硬件可执行指令）。CANN 向上支持 PyTorch、TensorFlow、MindSpore 等主流 AI 框架，向下通过驱动与昇腾 NPU 交互，将上层计算任务调度到硬件上高效执行。CANN 的存在使得上层框架无需关心 NPU 的底层细节。
- **第3层（框架与模型层）**：这一层提供对 AI 开发者友好的编程界面。MindSpore 是华为原生框架，与昇腾硬件深度协同；PyTorch/TensorFlow 则通过适配器（如 torch_npu）接入昇腾后端，让已有模型代码几乎零修改即可在 NPU 上运行。此外，**ATC**（Ascend Tensor Compiler，昇腾张量编译器）是将训练好的模型转换为昇腾 NPU 可执行的 `.om` 格式离线模型的必备工具，它属于 CANN 计算编译层的组成部分。
- **第4层（应用与工具层）**：AI 能力最终在这一层落地到具体场景。行业解决方案（如智能安防、图像识别、语音处理、智慧交通、医疗影像）、预训练模型库（提供开箱即用的模型）和 **MindStudio** 集成开发环境都属于这一层。MindStudio 提供工程管理、编译调试、性能分析等一站式开发能力。

### 各层次间的协同关系

昇腾软件体系的各层次之间有着紧密的协同关系：

- **硬件层 ↔ 操作系统与驱动层**：硬件层为上层提供了算力基础，而操作系统与驱动层解决了"如何让 CPU 与 NPU 协同工作"、"如何管理 NPU 设备资源"等基础问题。驱动是 CANN 与硬件通信的通道，CANN 通过驱动接口向下调用 NPU 执行计算。
- **CANN 的承上启下**：CANN 的底层通过驱动与硬件对接，上层则通过 AscendCL 编程接口向开发者开放。开发者在编写 AI 应用时调用的是 AscendCL 接口，而 AscendCL 的实现最终依赖于 CANN 的运行时和图编译器将计算任务调度到 NPU 上执行。
- **ATC 在模型部署阶段的作用**：开发者训练出的模型需要先通过 ATC 转换为 `.om` 格式，才能被 CANN 的运行时加载并在 NPU 上执行。
- **MindStudio 的全流程支撑**：MindStudio 开发工具为整个开发流程提供图形化的支撑环境，开发者可在其中完成代码编写、编译构建、模型转换（调用 ATC）、运行调试等工作。

> **章节导览**：按照从底层到上层的顺序，本章后续将展开——**昇腾操作系统**（3.2 节，包括操作系统构成及运行流程、Linux 内核、驱动开发、开发板接口、npu-smi 工具及系统构建方法，这些是 CANN 运行的基础支撑环境）和**昇腾应用软件开发**（3.3 节，包括应用程序的开发流程、远程开发方法及开发管理，这些是基于 CANN 的 AscendCL 编程接口进行应用开发的具体实践），帮助读者形成从硬件驱动到上层应用开发的完整技术路径认识。

### CANN 异构计算架构详解

<img src="../../images/cann_software_architecture.png" alt="CANN软件架构" style="display: block; margin-left: 0;" />

CANN（Compute Architecture for Neural Networks）是昇腾的核心软件栈，自底向上分为五层：

1. **计算基础层**：达芬奇架构 NPU 核心、内存控制器、PCIe 接口
2. **计算执行层**：Runtime 运行时、GE 图引擎、HCCL 通信库
3. **计算编译层**：图编译器（IR 中间表示）、TBE 引擎、算子融合
4. **计算服务层**：AOL 算子库（1400+ 优化算子）、AOE 调优引擎、框架适配器
5. **计算语言层**：AscendCL，提供统一 C/C++ API，屏蔽硬件差异

> **CANN 深入理解**：CANN 被称为整个体系的"心脏与大脑"。其中**图编译器**通过算子融合、常量折叠和数据布局转换等深度优化，将神经网络模型转换为高度优化的离线模型，使计算图被翻译成硬件直接执行的高效指令流。**AscendCL**（Ascend Computing Language）则提供了面向开发者的底层 C/C++ API，允许进行极致的硬件控制。正是 CANN 对硬件架构的深刻理解，才使得上层框架无需关心 NPU 的底层细节即可高效运行。
>
> **框架层深入理解**：华为原生的 MindSpore（昇思）框架与昇腾硬件达到了"血脉相通"的协同级别，支持动静统一的开发模式和自动并行等高级特性。通过"适配器"模式，PyTorch、TensorFlow 等主流框架也能无缝融入昇腾生态，开发者仅需简单修改代码即可将计算任务迁移到昇腾处理器，这种设计有效保护了开发生态和现有投入。
>
> **应用层深入理解**：全栈集成开发环境 MindStudio 是提升开发效率的"神器"，它集成了模型转换、性能分析、应用调试等工具，其性能分析器能够清晰地展示模型在执行时的算子耗时、数据搬运瓶颈等情况，是软硬件协同思想在开发流程中的集中体现。

**完整协同工作流**：从开发者在 Ubuntu 或 openEuler 系统上使用 MindSpore 或 PyTorch 编写模型，到通过 ATC 和 CANN 图编译器转换为高性能 `.om` 离线模型，再到通过 AscendCL 或框架接口在硬件上执行计算，最终通过 MindStudio 进行性能优化并打包成行业解决方案——这一完整的协同工作流，充分体现了昇腾软件体系如何通过各层次的精密配合，将原始的 AI 算力转化为切实的商业价值。

---

## 3. 用代码感受昇腾软件环境

在 Notebook 中，我们可以直接运行代码来感受昇腾平台的软件环境。下面逐步体验操作系统层、CANN 层的关键信息。

### 3.1 查看操作系统基础信息

下方代码通过 Python 标准库 `os`、`sys`、`platform` 获取当前运行环境的基础信息，包括 Python 版本、操作系统类型与版本、CPU 架构、逻辑核数和字节序等。

**预期结果**：在昇腾云沙箱环境中，架构应显示为 `aarch64`（ARM 64位），操作系统为 Linux；在本地 Windows 环境中运行时，架构会显示为 `AMD64`，这属于正常现象——说明当前并非在昇腾硬件上运行。

In [ ]:
import os, sys, platform

print("=" * 60)
print("昇腾平台操作系统环境信息")
print("=" * 60)
print(f"Python 版本:   {sys.version.split()[0]}")
print(f"操作系统:      {platform.system()} {platform.release()}")
print(f"机器架构:      {platform.machine()}")
print(f"处理器型号:    {platform.processor() or 'N/A'}")
print(f"CPU 逻辑核数:  {os.cpu_count()}")
print(f"进程字节序:    {sys.byteorder} ({'小端' if sys.byteorder=='little' else '大端'})")
print(f"Python 路径:   {sys.executable}")

**运行结果解读**：上述代码输出的信息能帮助我们快速判断当前运行环境。在昇腾云沙箱中，你会看到 `机器架构: aarch64`、`CPU 逻辑核数: 16` 等典型值；在本地环境中则可能看到 `AMD64` 架构和不同的核数。字节序 `little`（小端）是 ARM x86 平台的通用默认值。

### 3.2 查看内核版本与发行版信息

下方代码通过 `uname -r`、`uname -a`、`cat /etc/os-release` 和 `lscpu` 等 Linux 命令，查看当前系统的内核版本、发行版信息和 CPU 架构详情。这些信息反映了嵌入式 Linux 定制内核的特征。

**预期结果**：在昇腾环境中，内核版本通常为定制 LTS 版本（如 `5.10.0`），发行版为 Ubuntu 或 openEuler，架构为 `aarch64`。在 Windows 环境中，这些 Linux 命令不可用，会返回错误信息，属正常现象。

In [ ]:
# 查看内核版本与发行版信息 —— 感受嵌入式 Linux 定制内核
import subprocess

def run_cmd(cmd):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)
        return r.stdout.strip() if r.returncode == 0 else r.stderr.strip()
    except Exception as e:
        return f'[错误] {e}'

print("=" * 60)
print("[1] 内核版本 (uname -r)")
print("-" * 40)
print(run_cmd('uname -r'))

print("\n[2] 内核全部信息 (uname -a)")
print("-" * 40)
print(run_cmd('uname -a'))

print("\n[3] 发行版信息")
print("-" * 40)
print(run_cmd('cat /etc/os-release 2>/dev/null | head -5'))

print("\n[4] CPU 信息摘要")
print("-" * 40)
print(run_cmd('lscpu 2>/dev/null | grep -E "Architecture|CPU\\(s\\|Model name|Vendor" | head -4'))

### 3.3 检测 CANN 环境是否可用

下方代码尝试导入 `acl`（AscendCL）模块并调用 `acl.init()` 初始化 CANN 运行时。这是检测当前环境是否已配置 CANN 异构计算架构层的标准方法。

**预期结果**：在昇腾 CANN 环境中，会输出 `[OK] CANN (ACL) 环境可用！` 并显示 ACL 版本号，`CANN_AVAILABLE = True`；在未安装 CANN 的普通环境中，会输出 `[!] acl 模块未安装`，`CANN_AVAILABLE = False`。两种情况均属正常，后者仅说明当前不在昇腾硬件环境中。

In [ ]:
# 检测 CANN 环境是否可用 —— 感受异构计算架构层
CANN_AVAILABLE = False
try:
    import acl
    ret = acl.init()
    if ret == 0:
        CANN_AVAILABLE = True
        print("[OK] CANN (ACL) 环境可用！")
        # 查看版本
        try:
            major, minor, patch = acl.get_version()
            print(f"     ACL 版本: {major}.{minor}.{patch}")
        except Exception:
            pass
        acl.finalize()
    else:
        print(f"[!] ACL 初始化返回码: {ret}")
except ImportError:
    print("[!] acl 模块未安装，当前环境可能未配置 CANN")
except Exception as e:
    print(f"[!] CANN 环境异常: {e}")

print(f"\nCANN_AVAILABLE = {CANN_AVAILABLE}")

### 3.4 查看 NPU 硬件状态

下方代码调用 `npu-smi info` 命令查看昇腾 NPU 的型号、健康状态、温度、内存和利用率等信息。如果 `npu-smi` 不可用，则尝试通过 `torch_npu` 查询设备信息。

**预期结果**：在昇腾环境中，会输出类似 `npu-smi info` 的表格，显示 Ascend 910B3 设备的详细信息（健康状态 OK、利用率、温度、HBM 内存使用等）；在非昇腾环境中，会提示 `npu-smi 命令不可用` 和 `torch_npu 也未安装`，属正常现象。

In [ ]:
# 查看 NPU 硬件状态 —— 感受 NPU-SMI 系统管理工具
import subprocess

print("=" * 60)
print("npu-smi info  —— 昇腾 NPU 状态监控")
print("=" * 60)
r = subprocess.run('npu-smi info', shell=True, capture_output=True, text=True, timeout=15)
if r.returncode == 0 and r.stdout.strip():
    print(r.stdout)
else:
    print("[!] npu-smi 命令不可用，尝试使用 torch_npu 查询设备信息")
    try:
        import torch
        import torch_npu
        print(f"  NPU 可用:      {torch.npu.is_available()}")
        print(f"  NPU 卡数:      {torch.npu.device_count()}")
        print(f"  当前 NPU 设备: {torch.npu.current_device()}")
        print(f"  NPU 名称:      {torch.npu.get_device_name(0)}")
    except ImportError:
        print("  [!] torch_npu 也未安装，请在 CANN 环境中运行")

上面代码的运行结果展示了昇腾平台的三个关键层次：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">代码</th>
<th style="text-align: left;">对应层次</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>uname -r</code> / <code>lscpu</code></td>
<td style="text-align: left;">操作系统层</td>
<td style="text-align: left;">查看内核版本、CPU 架构（aarch64）、核数</td>
</tr>
<tr>
<td style="text-align: left;"><code>import acl</code></td>
<td style="text-align: left;">CANN 异构计算层</td>
<td style="text-align: left;">检测 ACL 运行时是否可用</td>
</tr>
<tr>
<td style="text-align: left;"><code>npu-smi info</code></td>
<td style="text-align: left;">系统管理工具</td>
<td style="text-align: left;">查看 NPU 型号、健康状态、温度、内存、利用率</td>
</tr>
</table>

**表格解读**：这三行代码分别对应昇腾四层架构中最关键的三个探测点。`uname -r` 和 `lscpu` 探测的是第1层（操作系统层），用于确认是否运行在 ARM aarch64 架构的定制内核上；`import acl` 探测的是第2层（CANN 异构计算架构层），确认 AscendCL 运行时是否已安装并可用；`npu-smi info` 探测的是系统管理工具，用于查看 NPU 硬件的实时状态。在真实昇腾环境中，三者均应成功返回信息；在非昇腾环境中，后两者会失败，这是判断运行环境的快速方法。

> **关键理解**：昇腾平台运行在 ARM aarch64 架构上，使用定制 Linux 内核，CANN 作为中间层连接上层框架与底层 NPU 硬件。若你在本地 Windows 或普通 Linux 上运行本 Notebook，`acl` 和 `npu-smi` 不可用是正常的——这说明当前环境没有昇腾硬件和 CANN 软件栈，后续涉及硬件操作的代码会进入仿真或提示模式。

---

## 4. 昇腾双轨制操作系统策略

<img src="./images/pptx_slide08_004.jpg" alt="双轨制操作系统" style="display: block; margin-left: 0;" />

昇腾平台采用 **Ubuntu + openEuler 双轨制**操作系统策略，两者定位互补：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">维度</th>
<th style="text-align: left;">Ubuntu</th>
<th style="text-align: left;">openEuler</th>
</tr>
<tr>
<td style="text-align: left;">定位</td>
<td style="text-align: left;">大众化入门 · 创新沙盒</td>
<td style="text-align: left;">企业级平台 · 生产环境</td>
</tr>
<tr>
<td style="text-align: left;">特点</td>
<td style="text-align: left;">卓越易用性、庞大社区、高效包管理</td>
<td style="text-align: left;">深度优化、多样性计算支持、开源社区协作</td>
</tr>
<tr>
<td style="text-align: left;">适用场景</td>
<td style="text-align: left;">学术研究、原型验证、开发调试</td>
<td style="text-align: left;">关键行业部署、高性能生产环境</td>
</tr>
</table>

**表格逐行解读**：
- **定位**：Ubuntu 定位为"大众化入门·创新沙盒"，强调让初学者快速上手、自由试验新想法；openEuler 定位为"企业级平台·生产环境"，强调高可靠性、高安全性和长期稳定运行。两者并非竞争关系，而是覆盖从学习到生产的完整生命周期。
- **特点**：Ubuntu 基于 Debian 发行版，凭借其深厚的桌面应用基因和对用户体验的极致追求，拥有全球最庞大的 Linux 社区之一，`apt` 包管理高效便捷，庞大的软件仓库使得 AI 开发所需的各种工具、库和依赖能够被轻松获取与部署，文档资源丰富；openEuler 由华为主导开源，专注于服务器、云计算、边缘计算等企业级场景，深度优化了多样性计算（x86/ARM/RISC-V），在内核调度、内存管理及网络栈等方面进行了增强，支持多核调度、热补丁等企业级特性。
- **适用场景**：学术研究、原型验证和开发调试阶段选用 Ubuntu，因为环境搭建快、第三方包丰富，其全球性活跃社区构成了巨大的知识库，几乎任何问题都能找到解决方案；关键行业部署（如金融、电力、交通）选用 openEuler，因为其经过严格测试、支持故障自愈和安全合规，为需要持续稳定运行的 AI 推理服务提供了坚实基础。

> **战略意义**：Ubuntu 降低技术门槛吸引全球开发者；openEuler 为关键行业提供高性能、高可靠性解决方案。两者"接地气"与"有底气"互补，使昇腾既能融入全球开源主流，又能服务于严苛的数字化核心场景。

下方代码通过读取 `/etc/os-release` 文件检测当前运行的 Linux 发行版，判断是 Ubuntu 还是 openEuler。

**预期结果**：在昇腾云沙箱中通常显示 Ubuntu 信息（含 `Ubuntu` 关键字）；在 openEuler 环境中则显示 openEuler 信息。在 Windows 或非 Linux 环境中会提示"无法读取"，属正常现象。

In [ ]:
# 检测当前运行的是哪种发行版
import subprocess

r = subprocess.run('cat /etc/os-release 2>/dev/null', shell=True, capture_output=True, text=True, timeout=5)
os_release = r.stdout.strip() if r.returncode == 0 else ''

print("当前系统发行版信息:")
print("-" * 40)
if os_release:
    for line in os_release.split('\n')[:6]:
        print(f"  {line}")
    if 'Ubuntu' in os_release:
        print("\n  => 当前为 Ubuntu 系统（创新沙盒定位）")
    elif 'openEuler' in os_release or 'EulerOS' in os_release:
        print("\n  => 当前为 openEuler 系统（企业级生产定位）")
    else:
        print("\n  => 其他 Linux 发行版")
else:
    print("  无法读取 /etc/os-release（可能不在 Linux 环境）")

---

## 5. 嵌入式 Linux 系统启动流程

嵌入式 Linux 系统的启动是一个**环环相扣、清晰分层**的过程，共四个阶段：

<img src="./images/pptx_slide11_005.png" alt="嵌入式Linux启动流程" style="display: block; margin-left: 0;" />

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">阶段</th>
<th style="text-align: left;">组件</th>
<th style="text-align: left;">职责</th>
<th style="text-align: left;">类比</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">BootLoader (U-Boot)</td>
<td style="text-align: left;">硬件初始化（时钟、内存、串口），从 Flash/eMMC 加载内核，传递启动参数</td>
<td style="text-align: left;">"开门人"——打开电源、检查设备、请内核出场</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">Linux Kernel</td>
<td style="text-align: left;">接管软硬件资源，进程调度、内存管理、中断处理、设备驱动，挂载根文件系统</td>
<td style="text-align: left;">"大管家"——统筹一切资源</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">Root Filesystem</td>
<td style="text-align: left;">系统库（glibc）、命令工具（BusyBox）、配置文件</td>
<td style="text-align: left;">"工具箱"——提供运行所需的工具和库</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">User Applications</td>
<td style="text-align: left;">后台服务、协议栈、GUI 交互程序，实现设备设计目的</td>
<td style="text-align: left;">"业务员"——干实际的活</td>
</tr>
</table>

**启动流程逐阶段解读**：
- **阶段1 BootLoader (U-Boot)**：上电后 CPU 最先执行的代码。U-Boot 负责最基本的硬件初始化——配置时钟频率、初始化 DDR 内存控制器、设置串口控制台（用于输出启动日志）。然后将 Flash/eMMC 中的 Linux 内核镜像加载到内存，并通过 bootargs 向内核传递启动参数（如根文件系统位置、控制台设备等），这如同将一封"说明书"交给了内核，告诉它该如何启动。U-Boot 是一款功能强大且完全开源的可移植引导加载程序，支持 ARM、MIPS、PowerPC 和 RISC-V 等几乎所有主流嵌入式处理器架构。类比"开门人"：开门、检查设备、请内核出场。
- **阶段2 Linux Kernel**：内核被加载后接管 CPU，开始初始化中断控制器、内存管理单元（MMU）、进程调度器等核心子系统，然后逐一加载设备树中描述的硬件驱动。最后挂载根文件系统并启动 init 进程（PID=1）。根文件系统是 Linux 系统赖以运行的基础环境，其中包含了操作系统运行所必需的所有程序、命令工具、系统库（如 glibc）和配置文件等。类比"大管家"：统筹一切软硬件资源。
- **阶段3 Root Filesystem**：根文件系统提供用户空间运行所需的一切：C 库（glibc/musl）、命令行工具（BusyBox 或 coreutils）、配置文件（/etc 目录）、设备节点（/dev 目录）。在嵌入式设备中，为了节省资源，通常使用 BusyBox 来提供一个精简而强大的命令工具集——一个二进制程序集成了上百个 Linux 命令。文件系统的格式也会根据存储硬件的不同进行选择，如针对 NAND Flash 的 UBIFS 或针对 eMMC 的 EXT4。没有根文件系统，内核启动后无处可去。类比"工具箱"：提供运行所需的工具和库。
- **阶段4 User Applications**：init 进程（通常是 /sbin/init）根据运行级别启动各项用户空间服务，如网络协议栈、SSH 守护进程、GUI 桌面环境、以及设备的核心业务程序。这些应用程序通过调用内核提供的系统接口和文件系统中的库文件来运行，最终让嵌入式设备实现其设计目的。类比"业务员"：干实际的活，实现设备的设计目的。

> **关键概念**：
> - **U-Boot**：功能强大且完全开源的引导加载程序，支持 ARM/MIPS/PowerPC/RISC-V 等主流架构
> - **BusyBox**：嵌入式设备中常用的精简命令工具集，一个二进制程序集成了上百个 Linux 命令，节省资源
> - **Init 进程**：内核启动的第一个用户空间进程（/sbin/init），是系统启动的"接力棒"

下方代码通过查看 init 进程（PID=1）、运行级别、内核启动参数和已挂载文件系统，来感受嵌入式 Linux 的启动流程。

**预期结果**：在 Linux 环境中，init 进程通常显示为 `systemd` 或 `init`，`/proc/cmdline` 包含 BootLoader 传递的启动参数（如 `root=...` 指定根文件系统设备），`mount` 输出显示 ext4/tmpfs/proc/sysfs 等已挂载的文件系统。在 Windows 环境中这些命令不可用，属正常现象。

In [ ]:
# 感受启动流程：查看 init 进程与系统运行级别
import subprocess

def run_cmd(cmd):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=5)
        return r.stdout.strip() if r.returncode == 0 else r.stderr.strip()
    except Exception as e:
        return f'[错误] {e}'

print("[1] init 进程 (PID=1，系统第一个用户空间进程)")
print("-" * 50)
print(run_cmd('ps -p 1 -o pid,comm 2>/dev/null || echo "无法查询"'))

print("\n[2] 当前运行级别")
print("-" * 50)
print(run_cmd('runlevel 2>/dev/null || who -r 2>/dev/null || echo "N/A"'))

print("\n[3] 内核启动参数 (BootLoader 传递给内核)")
print("-" * 50)
cmdline = run_cmd('cat /proc/cmdline 2>/dev/null')
print(cmdline[:200] + '...' if len(cmdline) > 200 else cmdline)

print("\n[4] 系统挂载的文件系统")
print("-" * 50)
print(run_cmd('mount 2>/dev/null | grep -E "ext4|tmpfs|proc|sysfs" | head -6'))

---

## 6. 昇腾定制 Linux 内核

昇腾平台使用**深度定制的 Linux 内核**，原因有三：

1. **硬件紧密关联**：Linux 内核与硬件平台紧密关联，昇腾专用 AI 硬件架构必须采用深度定制的内核
2. **充分发挥计算潜能**：定制内核集成专属驱动和硬件加速模块，充分发挥昇腾 AI 处理器的计算潜能
3. **上层软件栈基础**：定制内核是 CANN 等上层软件栈高效管理 NPU 的基石

Linux 内核已从一个特定平台的项目演变为一个支持多种处理器体系结构和硬件设备的通用内核。其成功的背后离不开开源协作模式，内核源代码由官方网站 kernel.org 统一发布，允许全球开发者自由使用、研究和修改。这种开放性正是厂商能够为特定硬件（如昇腾）进行二次开发的基础。

### 内核版本号构成

Linux 内核版本号由三部分组成：`VERSION.PATCHLEVEL.SUBLEVEL`（如 `6.10.0`），定义在源代码顶层目录的 Makefile 中。例如，数值 6、10 和 0 共同构成了版本 `6.10.0`。在昇腾的开发和部署环境中，通过命令查询或查验 Makefile 来确认所运行的是否为官方提供的、经过验证的定制内核版本，是确保整个 AI 应用稳定和高效的第一步。

> **LTS 版本**：在嵌入式系统领域，基于 LTS（长期支持）版本开发是行业最佳实践。如历史上重要的 `4.19`、`5.10` 以及较新的 `6.1` 等版本会获得长达数年的安全与维护更新，为嵌入式设备、工业控制系统及云服务器提供了至关重要的稳定性和安全性保障。

### 昇腾 310B 内核源码包结构

```
ascend-kernel/
├── kernel/      # LTS 内核源代码（含昇腾适配补丁）
├── driver/      # 昇腾310B 专用内核驱动模块
├── dtb/         # 设备树源码（描述硬件拓扑与配置）
├── scripts/     # 内核构建辅助脚本
├── config/      # 针对昇腾平台的内核配置文件
├── build/       # 编译生成目录（.ko 模块、Image 镜像）
├── build.sh     # 顶层构建脚本
└── abl/         # BootLoader 相关代码
```

- **kernel 目录**：包含经过修改和配置的、与特定 LTS 版本同步的官方 Linux 内核源代码，其中包含为适配昇腾芯片而加入的底层内核补丁和优化。
- **driver 目录**：存放昇腾 AI 处理器的专用内核驱动模块，负责实现操作系统与昇腾 310B 计算核心、内存及控制单元之间的通信和管理，是硬件能够被上层 CANN 调用的基础。
- **dtb 目录**：提供设备树源码文件，在内核启动阶段被加载，是实现跨平台支持的关键。
- **config 目录**：预置针对昇腾 310B 平台和不同应用场景的内核配置文件，构建时选择相应配置可确保生成的内核启用了所有必要的功能和驱动。
- **build.sh**：顶层构建脚本，为用户提供编译和打包内核的便捷入口，内部会调用标准的 Kbuild 系统。

> **设备树（Device Tree）**：以数据结构形式精确描述主处理器、内存、外设及昇腾 AI 处理器等硬件资源的拓扑和配置信息，在内核启动阶段被加载，实现驱动代码与具体硬件平台的解耦。

### 内核编译与更新流程

在主计算机上通过 `bash build.sh kernel` 命令配置并编译 Linux 内核。编译完成后，终端通常输出：
```
generate /opt/Ascend310B-source-opi/output/kernel_modules success!
generate /opt/Ascend310B-source-opi/output/Image success!
sign /opt/Ascend310B-source-opi/output/Image success!
```
编译后的 Image 文件存放于 `Ascend310B-source-opi/output` 目录下。然后执行如下命令更新 Image 文件，即更新了内核：
```bash
dd if=Image of=/dev/mmcblk1 count=61440 seek=32768 bs=512
```

In [ ]:
# 查看当前内核版本与已加载的昇腾相关内核模块
import subprocess

def run_cmd(cmd):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)
        return r.stdout.strip() if r.returncode == 0 else r.stderr.strip()
    except Exception as e:
        return f'[错误] {e}'

print("[1] 当前内核版本")
print("-" * 50)
print(run_cmd('uname -r'))

print("\n[2] 内核版本详细信息")
print("-" * 50)
print(run_cmd('uname -v'))

print("\n[3] 已加载的昇腾相关内核模块")
print("-" * 50)
mods = run_cmd('lsmod 2>/dev/null | grep -iE "ascend|davinci|npu|hispi|hisi" | head -10')
print(mods if mods else '(未检测到昇腾专用模块，或非 root 环境)')

print("\n[4] 设备树信息（如可用）")
print("-" * 50)
dt = run_cmd('ls /proc/device-tree 2>/dev/null | head -5')
print(dt if dt else '(设备树不可访问)')

---

## 7. Linux 操作系统驱动开发：三种方案对比

<img src="./images/pptx_slide14_006.jpg" alt="三种驱动方案" style="display: block; margin-left: 0;" />

Linux 驱动开发有三种主要方案，各有优劣：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">方案</th>
<th style="text-align: left;">核心机制</th>
<th style="text-align: left;">优势</th>
<th style="text-align: left;">挑战</th>
<th style="text-align: left;">适用场景</th>
</tr>
<tr>
<td style="text-align: left;">传统内核驱动</td>
<td style="text-align: left;">file_operations + 设备树 + 内核模块(.ko)</td>
<td style="text-align: left;">卓越性能、完整功能、标准框架</td>
<td style="text-align: left;">开发复杂、调试困难、维护成本高</td>
<td style="text-align: left;">SoC 专用加速器、复杂传感器融合</td>
</tr>
<tr>
<td style="text-align: left;">纯用户空间驱动</td>
<td style="text-align: left;">sysfs / libgpiod / UIO + mmap</td>
<td style="text-align: left;">开发简单、可用熟悉语言、错误隔离</td>
<td style="text-align: left;">上下文切换开销、功能受限</td>
<td style="text-align: left;">简单外设控制、快速原型验证</td>
</tr>
<tr>
<td style="text-align: left;">寄存器级驱动</td>
<td style="text-align: left;">ioremap + ioread32/iowrite32</td>
<td style="text-align: left;">极致性能、完全控制</td>
<td style="text-align: left;">最高难度、可移植性差、风险大</td>
<td style="text-align: left;">极端性能要求的特定场景</td>
</tr>
</table>

**三种方案逐行解读**：
- **传统内核驱动**：通过实现 `file_operations` 结构体（含 open/read/write/ioctl/release 等函数指针），将驱动注册为字符设备或块设备，在 `/dev` 目录下创建设备文件为用户空间提供标准化访问入口。驱动代码编译为 `.ko` 内核模块，可动态加载。设备树用于描述硬件信息（寄存器基地址、中断号等），实现驱动与硬件的解耦。优势在于性能卓越（直接内核态操作，无上下文切换开销，极低访问延迟）、功能完整（可处理中断、DMA 操作和精细内存管理）、能集成到各类内核标准框架中；挑战在于开发门槛高（需深入理解内核子系统）、调试困难（内核崩溃导致整机宕机）、部署和维护复杂。适用于 SoC 专用加速器、复杂传感器融合模块、网络设备等高复杂度、高性能场景。
- **纯用户空间驱动**：通过 sysfs（`/sys/class/gpio/` 目录下的 export、direction、value 文件）、libgpiod（`/dev/gpiochip*` 字符设备）或 UIO（Userspace I/O，通过 read/poll 在 `/dev/uioX` 上等待中断，mmap 映射设备寄存器）等接口，在用户空间直接操作硬件。优势在于开发简单（可用 Python/C 等熟悉语言和调试工具）、错误隔离（程序崩溃不影响内核稳定性）；挑战在于每次操作需系统调用（用户态与内核态切换开销）、难以实现复杂的 DMA 操作。适用于简单外设控制（LED、按键）、快速原型验证和低速数据采集。wiringOP 库即采用此方案。

  **现代 GPIO 用户空间库演进**：由于传统 sysfs 接口在性能和设计上存在缺陷（竞态条件、不支持中断轮询等），Linux GPIO 子系统已推出更先进的替代方案：
  - **libgpiod**：官方推荐并旨在取代旧 sysfs GPIO 接口的现代 C 库和工具集，基于新的 GPIO 字符设备接口，提供更清晰安全的 API（如 `gpiod_chip_open`、`gpiod_line_request_output`）和命令行工具（`gpioset`、`gpioget`），是当前用户空间操作 GPIO 的首选方案。
  - **librgpio**：更现代的 GPIO 库，同样构建在新的 GPIO 字符设备接口之上，提供友好的 API，体现社区向新标准的靠拢。
  - **pigpio**：功能丰富的库，不仅支持本地 GPIO 访问，还支持远程网络 GPIO 控制（适合分布式应用），提供硬件 PWM、波形生成和软件定时器等高级功能，在需要精确时序控制的应用（如驱动舵机）中非常受欢迎。

  > **新项目建议**：优先选择基于 GPIO 字符设备的 libgpiod，而非已被弃用的 sysfs 接口；需要高级功能或远程控制时，pigpio 是强大的补充选项。
- **寄存器级驱动**：本质上是传统内核驱动的特殊实现形式，通过 `of_iomap` 或 `ioremap()` 将设备寄存器的物理地址映射到内核虚拟地址空间，再用 `ioread32()`/`iowrite32()` 等专用函数直接读写寄存器（严禁普通指针解引用，以确保正确处理具有副作用的设备寄存器）。开发者必须深入理解芯片参考手册中每个控制寄存器、状态寄存器和数据寄存器的位字段定义。优势在于极致性能（消除中间抽象层开销）和完全控制（支持标准抽象层无法达成的特殊操作）；挑战在于最高难度（需查阅芯片手册、精确计算寄存器偏移）、可移植性极差（换芯片需重写）、风险大（错误操作可能导致设备损坏或系统崩溃）。仅适用于极端性能要求的特定场景，如高频金融交易系统网卡驱动、专用硬件加速器驱动。

> **技术选型建议**：纯用户空间驱动以卓越简易性成为快速原型开发和简单控制的首选；传统内核驱动在性能、功能和集成度方面实现最佳平衡，是大多数正规复杂驱动开发的标准选择；寄存器级驱动则为专家级开发者在特定场景下追求极致性能提供可能。实际项目需从开发效率、性能要求、硬件复杂性和长期维护成本等多维度综合权衡。

<img src="../../images/driver.jpg" alt="驱动开发" width="600px" style="display: block; margin-left: 0;" />

下方代码通过 sysfs 和 /dev 设备节点来体验用户空间驱动方案，检查 GPIO、SPI、I2C 等子系统的设备节点是否可用。

In [ ]:
# 体验用户空间驱动：通过 sysfs 查看 GPIO 子系统
import subprocess, os

def run_cmd(cmd):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=5)
        return r.stdout.strip() if r.returncode == 0 else r.stderr.strip()
    except Exception as e:
        return f'[错误] {e}'

print("[1] GPIO 子系统 (sysfs 接口)")
print("-" * 50)
print(run_cmd('ls /sys/class/gpio/ 2>/dev/null || echo "sysfs gpio 不可访问"'))

print("\n[2] GPIO 字符设备 (libgpiod 接口)")
print("-" * 50)
print(run_cmd('ls /dev/gpiochip* 2>/dev/null || echo "gpiochip 设备不存在"'))

print("\n[3] SPI 设备节点")
print("-" * 50)
print(run_cmd('ls /dev/spidev* 2>/dev/null || echo "SPI 设备节点不存在"'))

print("\n[4] I2C 设备节点")
print("-" * 50)
print(run_cmd('ls /dev/i2c-* 2>/dev/null || echo "I2C 设备节点不存在"'))

print("\n[5] 已加载的驱动模块总数")
print("-" * 50)
print(run_cmd('lsmod 2>/dev/null | tail -n +2 | wc -l'))

**运行结果解读与设备节点缺失原因分析**：

上述代码检查了 GPIO、SPI、I2C 三种外设子系统的设备节点。在真实昇腾香橙派开发板上运行时，`/sys/class/gpio/` 下会有 `export`、`unexport` 及 `gpiochip*` 设备，`/dev/spidev*` 和 `/dev/i2c-*` 节点也会存在。但在本 Notebook 运行环境中（昇腾云沙箱或本地非开发板环境），你可能看到如下结果：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">检测项</th>
<th style="text-align: left;">云沙箱/本地环境结果</th>
<th style="text-align: left;">真实香橙派结果</th>
</tr>
<tr>
<td style="text-align: left;">GPIO sysfs</td>
<td style="text-align: left;"><code>export unexport</code>（仅基础接口）</td>
<td style="text-align: left;"><code>export unexport gpiochip0 ...</code></td>
</tr>
<tr>
<td style="text-align: left;">GPIO 字符设备</td>
<td style="text-align: left;"><code>gpiochip 设备不存在</code></td>
<td style="text-align: left;"><code>/dev/gpiochip0</code> 等设备节点</td>
</tr>
<tr>
<td style="text-align: left;">SPI 设备节点</td>
<td style="text-align: left;"><code>SPI 设备节点不存在</code></td>
<td style="text-align: left;"><code>/dev/spidev0.0</code> 等设备节点</td>
</tr>
<tr>
<td style="text-align: left;">I2C 设备节点</td>
<td style="text-align: left;"><code>I2C 设备节点不存在</code></td>
<td style="text-align: left;"><code>/dev/i2c-0</code> 等设备节点</td>
</tr>
<tr>
<td style="text-align: left;">驱动模块总数</td>
<td style="text-align: left;"><code>0</code></td>
<td style="text-align: left;">数十个已加载模块</td>
</tr>
</table>

**为什么在云沙箱/本地环境中看不到设备节点？** 原因如下：

1. **硬件差异**：云沙箱环境提供的是昇腾 NPU 算力（Atlas A2 服务器），并非香橙派开发板。香橙派上的 GPIO/SPI/I2C 控制器是 SoC 内置的外设控制器，云沙箱服务器上没有这些外设控制器硬件，因此内核不会创建对应的设备节点。
2. **内核配置差异**：香橙派的定制内核启用了 `CONFIG_GPIO_SYSFS`、`CONFIG_SPI_SPIDEV`、`CONFIG_I2C_CHARDEV` 等编译选项，而云沙箱的服务器内核可能未启用这些选项（因为服务器不需要 GPIO/SPI/I2C 外设）。
3. **设备树差异**：香橙派的设备树（DTS）中描述了 GPIO 控制器、SPI 控制器和 I2C 控制器的硬件地址和引脚复用信息，内核启动时根据设备树加载对应驱动并创建设备节点。云沙箱的设备树中不包含这些外设描述。
4. **权限限制**：即使存在部分设备节点，非 root 用户也可能无权访问 `/dev/` 下的设备文件。

> **学习建议**：此结果完全正常，不影响对驱动开发概念的理解。要体验真实的 GPIO/SPI/I2C 设备节点操作，需要将代码部署到香橙派 Orange Pi AI Pro 开发板上运行。后续实验（02、03）中提供了仿真模式，可在无硬件环境下模拟这些外设操作。

---

## 8. 昇腾开发板接口开发：wiringOP 库

<img src="./images/pptx_slide18_007.jpg" alt="wiringOP库" style="display: block; margin-left: 0;" />

wiringOP 是香橙派官方提供的 GPIO/SPI/I2C/PWM 控制库，对标树莓派的 wiringPi，提供 Arduino 风格的直观 API。

### 核心实现机制

wiringOP 库的实现原理与树莓派的 wiringPi 库高度相似，其核心在于通过系统调用和文件操作来实现对硬件资源的用户空间访问。具体来说，wiringOP 的实现建立在 Linux 内核提供的 GPIO、PWM、SPI 等子系统之上，通过操作这些子系统在 `/sys` 和 `/dev` 目录下暴露的设备文件来完成硬件控制。

- **基于 Linux 子系统**：通过操作 `/sys` 和 `/dev` 目录下的设备文件完成硬件控制
- **GPIO 控制**：调用 `pinMode()` 函数设置引脚方向时，库内部会向 `/sys/class/gpio/export` 文件写入引脚编号，使该引脚在 sysfs 中可见，然后向对应的 `/sys/class/gpio/gpioX/direction` 文件写入 "in" 或 "out" 来配置输入输出模式。`digitalWrite()` 和 `digitalRead()` 分别通过读写 `/sys/class/gpio/gpioX/value` 文件来实现电平读写
- **PWM 控制**：调用 `pwmWrite()` 时，库函数操作 `/sys/class/pwm/pwmchipX` 目录下的 export、period、duty_cycle 和 enable 等文件，通过设置周期、占空比和使能参数来生成 PWM 波形。支持硬件 PWM（依赖芯片专用 PWM 控制器）和软件 PWM（通过定时器中断模拟）两种模式
- **SPI/I2C 通信**：通过操作 `/dev/spidevX.X` 和 `/dev/i2c-X` 设备文件，使用 `ioctl()` 系统调用进行设备配置和数据传输，封装了底层复杂的参数设置过程
- **中断处理**：通过 `select()` / `poll()` 系统调用监听 `/sys/class/gpio/gpioX/value` 文件的状态变化，一旦检测到电平跳变就触发用户注册的回调函数。虽然不能达到真正的硬件中断响应速度，但对大多数应用场景已经足够

### 常用命令

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">命令</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>gpio readall</code></td>
<td style="text-align: left;">查看所有引脚映射对照表（含 BCM/wPi 编号、名称、模式、电平）</td>
</tr>
<tr>
<td style="text-align: left;"><code>gpio mode 2 out</code></td>
<td style="text-align: left;">设置 wPi 序号为 2 的引脚为输出模式</td>
</tr>
<tr>
<td style="text-align: left;"><code>gpio qmode 2</code></td>
<td style="text-align: left;">查询引脚当前模式（输出 OUT 或输入 IN）</td>
</tr>
<tr>
<td style="text-align: left;"><code>gpio write 2 1</code></td>
<td style="text-align: left;">设置引脚高电平（1=高，0=低）</td>
</tr>
<tr>
<td style="text-align: left;"><code>gpio write 2 0</code></td>
<td style="text-align: left;">设置引脚低电平</td>
</tr>
<tr>
<td style="text-align: left;"><code>gpio read 2</code></td>
<td style="text-align: left;">读取引脚当前电平状态</td>
</tr>
</table>

**命令表解读**：`gpio readall` 是最常用的命令，它输出香橙派 40Pin 排针的完整引脚映射表，包含 BCM 编号、wiringPi 编号、引脚名称、模式和电平值，是验证 wiringOP 安装是否成功的标准方法。`gpio mode 2 out` 将 wiringPi 编号为 2 的引脚设置为输出模式。`gpio qmode 2` 查询引脚当前模式。`gpio write 2 1` 和 `gpio write 2 0` 分别设置高电平和低电平，可用万用表测量电压（高电平约 3.3V，低电平约 0V）。`gpio read 2` 读取当前电平状态。这组命令采用 Arduino 风格的简洁语法，降低了 GPIO 操作的编程门槛。

### C 语言 API 示例

在程序中也可通过 `digitalWrite()`、`digitalRead()`、`pinMode()` 等函数控制外部硬件，以 GPIO7_02（wPi 序号 2）为例：

```c
pinMode(2, OUTPUT);       // 设置 GPIO7_02 为输出
digitalWrite(2, HIGH);    // 设置高电平
digitalRead(2);           // 读取引脚状态
```

> **性能考量**：由于 wiringOP 运行在用户空间，每次硬件操作都需要经过用户态到内核态的上下文切换，会引入额外的延迟和 CPU 开销。对于需要精确时序控制或高实时性的应用场景，这种延迟可能成为瓶颈。但总体而言，wiringOP 特别适合快速原型开发、教学演示和中等复杂度的嵌入式项目，是昇腾香橙派生态中最为流行的硬件编程库。

---

## 9. 昇腾 NPU-SMI 系统管理工具

<img src="./images/pptx_slide21_008.jpg" alt="NPU-SMI工具" style="display: block; margin-left: 0;" />

NPU-SMI 是昇腾 AI 处理器的标准管理工具，功能定位类似 NVIDIA 的 `nvidia-smi`，用于监控、管理和维护 NPU 硬件设备。

### 核心功能

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">功能类别</th>
<th style="text-align: left;">具体功能</th>
<th style="text-align: left;">命令示例</th>
</tr>
<tr>
<td style="text-align: left;">信息监控</td>
<td style="text-align: left;">设备概览、健康指标、算力利用率</td>
<td style="text-align: left;">npu-smi info</td>
</tr>
<tr>
<td style="text-align: left;">使用率查询</td>
<td style="text-align: left;">Aicore/Aicpu 利用率、内存带宽</td>
<td style="text-align: left;">npu-smi info -t usages -i 0 -c 0</td>
</tr>
<tr>
<td style="text-align: left;">设备管理</td>
<td style="text-align: left;">硬件复位、功耗管理、温度保护</td>
<td style="text-align: left;">npu-smi set ...</td>
</tr>
<tr>
<td style="text-align: left;">CPU 配置</td>
<td style="text-align: left;">Control CPU / AI CPU 分配</td>
<td style="text-align: left;">npu-smi info -t cpu-num-cfg -i 0 -c 0</td>
</tr>
</table>

**功能表逐行解读**：
- **信息监控**：`npu-smi info` 是最基础也是最常用的命令，输出所有 NPU 设备的概览表格，包括设备数量、物理位置、芯片型号（如 Ascend-910B3）、固件版本、健康状态（OK/Warning/Alarm）、功率、温度、HBM/DDR 内存使用量和 AI Core 利用率。这些数据对于性能调优和瓶颈分析具有重要价值。类似 `nvidia-smi` 的作用。
- **使用率查询**：`npu-smi info -t usages -i 0 -c 0` 查询指定设备（`-i 0`）和芯片（`-c 0`）的详细使用率，包括 Aicore（AI 计算核心）和 Aicpu（AI 控制CPU）的利用率、内存带宽使用情况、Hugepages 使用率等，用于性能调优和瓶颈分析。
- **设备管理**：`npu-smi set` 系列命令用于硬件复位（清除设备当前状态并恢复至初始就绪状态）、设置功耗上限和温度保护阈值等管理操作。支持多种电源模式的切换（高性能模式与节能模式），当芯片温度超过预设值时自动触发保护机制。通常需要 root 权限，在生产环境中用于设备维护。
- **CPU 配置**：`npu-smi info -t cpu-num-cfg -i 0 -c 0` 查询或设置 SoC 内部 Control CPU 与 AI CPU 的分配比例，这是昇腾 SoC 的特色功能，可根据工作负载特性动态调整通用计算与 AI 计算的资源分配。

**其他关键功能**：
- **任务管理**：NPU-SMI 具备完善的进程监控功能，可列出当前正在使用 NPU 计算资源的所有运行中进程，详细显示每个进程的 PID、所属用户、占用 NPU 内存大小、计算核心使用情况等信息。当出现进程异常或需要释放资源时，管理员可直接通过工具终止指定进程，这在共享开发环境或生产服务器中尤为重要。
- **系统维护**：集成强大的日志收集和诊断功能，可一键获取与 NPU 相关的所有系统日志、驱动日志和运行日志，对于分析硬件故障、驱动兼容性问题或应用程序错误具有关键作用。同时提供硬件自检功能（包括内存测试、计算核心测试等），确保硬件在投入使用时处于正常状态。
- **使用模式**：既支持单次查询模式（通过命令行参数获取特定信息），也提供实时监控模式（交互式界面持续观察系统状态变化），适用于从快速系统检查到长期运行监控的各种场景。

下方代码调用 `npu-smi info` 及其子命令，查询 NPU 设备基本信息、使用率详情和 CPU 配置信息。

**预期结果**：在昇腾环境中，会输出 NPU 设备概览表（含设备 ID、名称、健康状态、温度、HBM 内存、利用率等）、Aicore/Aicpu 使用率详情和 Control CPU/AI CPU 分配配置。在非昇腾环境中，三个查询均显示"不可用"，属正常现象。

In [ ]:
# NPU-SMI 详细信息查询
import subprocess

def run_npu_smi(args=''):
    cmd = f'npu-smi info {args}'.strip()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=15)
    return r.stdout.strip() if r.returncode == 0 else None

print("[1] NPU 设备基本信息")
print("-" * 50)
out = run_npu_smi()
print(out if out else '(npu-smi 不可用)')

print("\n[2] NPU 使用率详情")
print("-" * 50)
out = run_npu_smi('-t usages -i 0 -c 0')
print(out if out else '(使用率查询不可用)')

print("\n[3] CPU 配置信息")
print("-" * 50)
out = run_npu_smi('-t cpu-num-cfg -i 0 -c 0')
print(out if out else '(CPU 配置查询不可用)')

### 昇腾 SoC CPU 分配策略

昇腾 SoC 内部的 CPU 分为两类：

- **Control CPU（控制 CPU）**：负责系统控制、任务调度等通用计算任务
- **AI CPU（AI 计算 CPU）**：专门用于执行 AI 算子计算任务

默认配置为 `0:3:1`（3 个 Control CPU + 1 个 AI CPU），总 CPU 数为 4。配置参数格式为 `data_cpu:ctrl_cpu:ai_cpu`。

**实际运行现象**：当 Linux 系统跑满后，使用 `htop` 命令会看到有一个 CPU 的占用率始终接近 0——这是正常的，因为该 CPU 默认用于 AI CPU。如果当前环境模型中无 AI CPU 算子，且运行业务时查询 AI CPU 占用率持续为 0，则可以将 AI CPU 的数量配置为 0：

```bash
# 查询 AI CPU 占用率
npu-smi info -t usages -i 0 -c 0

# 将 4 个 CPU 都设置为 Control CPU（0:4:0），需重启生效
sudo npu-smi set -t cpu-num-cfg -i 0 -c 0 -v 0:4:0
```

设置完成后，再运行任务让所有 CPU 跑满，`htop` 命令就能看到 4 个 CPU 的占用率都能达到 100%。

---

## 10. Linux 系统构建方法

<img src="./images/pptx_slide24_009.png" alt="Linux系统构建方法" style="display: block; margin-left: 0;" />

嵌入式 Linux 系统有三种主流构建方法：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">方法</th>
<th style="text-align: left;">定位</th>
<th style="text-align: left;">特点</th>
<th style="text-align: left;">适用场景</th>
</tr>
<tr>
<td style="text-align: left;">Buildroot</td>
<td style="text-align: left;">轻量级方案</td>
<td style="text-align: left;">Kconfig 配置、高效构建、镜像小</td>
<td style="text-align: left;">中小型项目、快速原型开发</td>
</tr>
<tr>
<td style="text-align: left;">Yocto</td>
<td style="text-align: left;">工业级方案</td>
<td style="text-align: left;">BitBake 引擎、分层机制、可重复构建</td>
<td style="text-align: left;">汽车电子、工业自动化、IoT 基础设施</td>
</tr>
<tr>
<td style="text-align: left;">手动构建</td>
<td style="text-align: left;">极致定制方案</td>
<td style="text-align: left;">从工具链开始逐步搭建、最高灵活性</td>
<td style="text-align: left;">追求极致控制的专家级开发</td>
</tr>
</table>

**构建方法逐行解读**：
- **Buildroot**：轻量级嵌入式 Linux 构建系统，使用 Kconfig 菜单配置（类似内核 `make menuconfig`），自动下载、交叉编译内核、引导加载程序和根文件系统，生成完整可烧录的镜像。优势是构建速度快、生成的镜像小（可低至几 MB）、学习门槛低；劣势是定制灵活性有限、缺乏包管理（不支持在线安装软件）。适合中小型项目和快速原型开发。
- **Yocto**：工业级 Linux 构建框架，使用 BitBake 构建引擎和分层（Layer）机制组织代码。每一层可独立维护和复用，支持可重复构建（相同输入必定产生相同输出，满足审计要求）。优势是高度灵活、支持完整的包管理（opkg/rpm/deb）、社区生态丰富；劣势是学习曲线陡峭、构建时间长。是汽车电子、工业自动化和 IoT 基础设施的首选方案。
- **手动构建**：从交叉编译工具链（gcc、glibc）开始，手动编译 U-Boot、内核、根文件系统每个组件。优势是最高灵活性（每个组件都可精确控制）和最小体积；劣势是难度最高、容易出错、难以维护。仅适合追求极致控制的专家级开发。

### 昇腾系统镜像三种版本

- **最小镜像（Minimal）**：可启动但缺少部分依赖，适合最小化系统体积
- **完整镜像（Complete）**：包含所有必要软件包和依赖，适合大多数开发场景
- **压缩扩容镜像（Compress）**：支持动态扩容，适合需要灵活存储管理的场景

**镜像版本选择建议**：初学者建议从完整镜像开始，避免因缺少依赖而反复排错；对存储空间敏感的嵌入式产品可选用最小镜像，按需添加软件包；需要灵活调整分区大小的场景（如开发阶段频繁安装软件）可选用压缩扩容镜像。

### base.sh 构建脚本设计

昇腾香橙派开发板采用专门定制的系统构建框架，其核心是 `base.sh` 脚本——一个用于为昇腾 AI 处理器开发板构建和打包定制 Linux 系统镜像的自动化工具。其核心设计理念是**模块化与可配置性**：

- **配置驱动执行**：脚本本身作为"框架"，不直接包含构建步骤，而是通过解析外部配置文件 `cfg.json`（定义功能列表和开关）和函数库 `func.sh`（包含具体实现），动态决定执行流程。这种设计将"做什么"（配置文件）与"怎么做"（函数库）分离，极具灵活性。
- **执行流程**：权限检查 → 参数检查 → 依赖安装 → 配置解析 → 功能执行。每成功执行一个功能，脚本会在配置文件中将其标记为已完成（状态持久化），中途失败后下次运行可跳过已完成步骤。
- **核心功能**：日志函数（带时间戳统一记录）、命令执行辅助函数（自动捕获失败并终止）、依赖下载函数（先查本地缓存，再按 URL 自动下载）、错误与中断处理函数（Ctrl+C 时清理资源）。
- **系统依赖**：自动安装 `jq`（JSON 处理）、`qemu-user-static`（x86 主机上运行 ARM64 程序）、`parted`/`kpartx`（磁盘分区管理）等工具。

> **设计精髓**：这种基于配置的动态执行模式，使构建流程完全由数据（配置文件）驱动，无需改动脚本核心逻辑即可适配不同 Linux 发行版、版本号和功能需求。

---

## 11. 应用程序开发流程：从 Hello World 到远程开发

<img src="./images/pptx_slide27_010.jpg" alt="Hello World开发" style="display: block; margin-left: 0;" />

### Hello World 开发流程

1. **环境准备**：`uname -m`（预期 aarch64）、`gcc --version`
2. **创建项目**：`mkdir hello_world && cd hello_world`
3. **编写代码**：创建 `hello.c`
4. **编译**：`gcc -o hello hello.c`
5. **验证**：`file hello`（预期 ELF 64-bit LSB executable, ARM aarch64）
6. **运行**：`./hello`

### Makefile 管理编译

对于单个文件，直接使用 GCC 命令很简单。但当项目稍复杂时，使用 Makefile 管理编译过程是更专业的做法：

```makefile
CC = gcc              # 定义编译器
CFLAGS = -Wall        # 定义编译选项
TARGET = hello        # 定义目标可执行文件名称
SRC = hello.c         # 定义源文件

$(TARGET): $(SRC)
	$(CC) $(CFLAGS) -o $(TARGET) $(SRC)

clean:
	rm -f $(TARGET)

.PHONY: clean
```

使用 Makefile 时，编译程序只需输入 `make`；清理生成的可执行文件，输入 `make clean`。

### 远程开发模式：VSCode Remote-SSH

通过 SSH 协议连接到昇腾开发板，在主机 VSCode 中直接编辑开发板上的代码，使用集成终端编译运行。核心思想是**"职责分离"**：主机负责编辑交互，开发板负责算力执行。

**工作原理**：安装 Remote-SSH 扩展后，VSCode 会在后台通过 SSH 连接，自动在昇腾开发板上部署一个轻量级的服务器端进程。该进程负责处理代码编辑、文件管理和扩展功能等请求，而主机上的 VSCode 界面则作为高效的客户端进行交互。

**带来的体验提升**：开发者可在熟悉的主机 VSCode 环境中，直接打开、浏览和编辑存储在昇腾开发板上的项目代码，继续使用智能语法高亮、自动代码补全、集成 Git 版本控制以及图形化调试器等强大功能。需要编译运行时，在 VSCode 内部集成的终端（本质是 SSH 会话）中输入命令，所有命令都在开发板原生环境中执行。

**完整工作流**：通过 VSCode Remote-SSH 连接开发板 → 图形化界面编写修改代码 → 编辑器内一键打开终端编译运行 → 需要诊断时另开 SSH 终端运行 `npu-smi` 等监控命令。整个开发、调试和测试循环在集成环境中完成。

下方代码在 Notebook 中模拟完整的嵌入式 C 程序开发流程：检查编译器、检查架构、编写 hello.c、编译、验证可执行文件格式、运行。

**预期结果**：在 Linux 环境中，会输出 gcc 版本信息、架构（`aarch64` 或 `x86_64`），编译成功后 `file` 命令显示 `ELF 64-bit LSB executable, ARM aarch64`（昇腾环境）或 `ELF 64-bit LSB executable, x86-64`（普通 Linux），最后输出 `Hello, World from Ascend!`。在 Windows 环境中 gcc 不可用，编译会失败，属正常现象。

In [ ]:
# Hello World —— 在 Notebook 中体验嵌入式 C 程序开发流程
import subprocess, os, tempfile

print("[1] 检查编译器")
print("-" * 50)
r = subprocess.run('gcc --version 2>/dev/null | head -1', shell=True, capture_output=True, text=True)
print(r.stdout.strip() if r.stdout.strip() else 'gcc 不可用')

print("\n[2] 检查系统架构 (预期 aarch64)")
print("-" * 50)
r = subprocess.run('uname -m', shell=True, capture_output=True, text=True)
arch = r.stdout.strip()
print(f"架构: {arch}")

# 尝试编译并运行 Hello World
hello_c = '#include <stdio.h>\nint main() { printf("Hello, World from Ascend!\\n"); return 0; }\n'
tmpdir = tempfile.mkdtemp()
src_path = os.path.join(tmpdir, 'hello.c')
bin_path = os.path.join(tmpdir, 'hello')
with open(src_path, 'w') as f:
    f.write(hello_c)

print("\n[3] 编译 Hello World")
print("-" * 50)
r = subprocess.run(f'gcc -o {bin_path} {src_path}', shell=True, capture_output=True, text=True)
if r.returncode == 0:
    print("编译成功!")
    print("\n[4] 验证可执行文件格式")
    print("-" * 50)
    r2 = subprocess.run(f'file {bin_path}', shell=True, capture_output=True, text=True)
    print(r2.stdout.strip())
    print("\n[5] 运行程序")
    print("-" * 50)
    r3 = subprocess.run(f'{bin_path}', shell=True, capture_output=True, text=True)
    print(r3.stdout.strip())
else:
    print(f"编译失败: {r.stderr.strip()}")
    print("(在非 Linux 环境或无 gcc 时属正常现象)")

---

## 12. 代码版本管理：Git

<img src="./images/pptx_slide30_011.jpg" alt="Git版本管理" style="display: block; margin-left: 0;" />

Git 是分布式版本控制系统，提供版本控制、变更追踪、分支管理、回退处理等七大核心功能。

### Git 生态系统

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">平台</th>
<th style="text-align: left;">定位</th>
<th style="text-align: left;">特点</th>
</tr>
<tr>
<td style="text-align: left;">Git</td>
<td style="text-align: left;">底层引擎</td>
<td style="text-align: left;">分布式版本控制核心</td>
</tr>
<tr>
<td style="text-align: left;">GitHub</td>
<td style="text-align: left;">全球开源社区</td>
<td style="text-align: left;">Fork/PR 机制促进协作</td>
</tr>
<tr>
<td style="text-align: left;">GitLab</td>
<td style="text-align: left;">企业级 DevOps</td>
<td style="text-align: left;">CI/CD、代码审查、私有部署</td>
</tr>
<tr>
<td style="text-align: left;">Gitee</td>
<td style="text-align: left;">国内本土化</td>
<td style="text-align: left;">国内团队协作友好</td>
</tr>
</table>

**Git 生态表解读**：Git 本身是命令行工具（底层引擎），提供 `init/add/commit/branch/merge/log` 等核心操作。GitHub 是全球最大的开源代码托管平台，通过 Fork（派生个人副本）和 PR（拉取请求）机制促进全球协作。GitLab 定位企业级，内置 CI/CD 流水线、代码审查和私有部署能力，适合企业内部使用。Gitee（码云）是国内本土化平台，访问速度快、符合国内合规要求，适合国内团队协作。

### 个人项目代码维护

对于个人开发者，Git 的主要目的是对代码版本进行系统性维护，记录每个关键节点：

1. **项目初始化**：`git init` 初始化本地仓库，`git add .` 添加文件到暂存区，`git commit -m "feat: 项目初始化"` 首次提交
2. **日常开发**：遵循"小步快走"策略，每完成一个小功能点就提交一次。提交信息使用约定前缀：`feat:`（新功能）、`fix:`（修复缺陷）
3. **版本标签**：重大进展时创建标签：`git tag -a v1.0.0 -m "模型准确率达标首版"`，用 `git tag` 列出所有里程碑
4. **代码回溯**：`git status` 查看修改情况，`git checkout -- [文件]` 还原临时修改，`git reset --hard [commit_id]` 回退到稳定时间点
5. **分支运用**：`git checkout -b experiment/transformer-arch` 创建实验分支，成功则合并回主分支，失败则直接删除

### 团队项目代码维护

1. **项目初始化与托管**：项目经理 `git init` 初始化仓库，在 GitHub 创建远程仓库，`git remote add origin <URL>` 关联，`git push -u origin main` 推送基础代码
2. **并行开发**：采用功能分支工作流，`git checkout -b feature/new-cnn-model` 创建专属分支，在独立分支上安全修改和频繁提交
3. **代码合并与协作**：开发完成后 `git push origin feature/new-cnn-model`，在 GitHub 发起 Pull Request（PR），团队成员在 PR 中审查代码、提出修改建议，发起者根据反馈继续修改
4. **版本发布**：审查通过后合并到 main 分支，`git tag -a v1.0.0 -m "首个稳定版"` 创建标签，`git push origin v1.0.0` 推送到远程
5. **问题修复**：基于稳定标签创建热修复分支 `git checkout -b hotfix-issue v1.0.0`，修复后合并回 main；紧急情况可用 `git revert <commit_id>` 直接撤销问题提交

> **Git 核心价值**：为项目构建完整的"代码演进档案"，赋予开发者大胆重构和试验的信心——因为所有的"过去"都被妥善保存着，随时可"重返"。在团队协作中，Pull Request 机制确保了每次代码变更都经过审查，保证代码质量。

下方代码查看 Git 版本和用户配置，并输出常用 Git 命令速查表。

**预期结果**：会输出 Git 版本号（如 `git version 2.34.1`）、用户名和邮箱配置（可能为"未设置"），以及一组常用 Git 命令及其功能说明。

In [ ]:
# 查看 Git 版本与基本配置
import subprocess

def run_cmd(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=5)
    return r.stdout.strip() if r.returncode == 0 else r.stderr.strip()

print("[1] Git 版本")
print("-" * 40)
print(run_cmd('git --version'))

print("\n[2] Git 用户配置")
print("-" * 40)
print(f"  user.name:  {run_cmd('git config user.name') or '(未设置)'}")
print(f"  user.email: {run_cmd('git config user.email') or '(未设置)'}")

print("\n[3] 常用 Git 命令速查")
print("-" * 40)
commands = [
    ('git init',              '初始化仓库'),
    ('git add .',             '添加到暂存区'),
    ('git commit -m "msg"',  '提交变更'),
    ('git log --oneline',     '查看提交历史'),
    ('git branch <name>',     '创建分支'),
    ('git checkout -b <name>','创建并切换分支'),
    ('git merge <name>',      '合并分支'),
    ('git tag -a v1.0 -m msg','创建标签'),
]
for cmd, desc in commands:
    print(f'  {cmd:30s} # {desc}')

---

## 13. 用 PyTorch + torch_npu 感受昇腾异构计算

最后，通过一个完整的矩阵乘法示例，感受昇腾平台上 Host（CPU）与 Device（NPU）的异构协作——这正是嵌入式操作系统与 CANN 协同工作的体现。

<img src="../../images/host_and_device.png" alt="Host-Device异构协作" style="display: block; margin-left: 0;" />

**Host-Device 异构协作模型**：在昇腾平台上，Host（CPU/主机）负责逻辑控制、数据准备和任务调度，Device（NPU/设备）负责大规模并行计算。数据在 Host 和 Device 之间通过 PCIe 总线传输，计算任务由 CANN 运行时调度到 NPU 执行。这种"职责分离"的架构正是嵌入式操作系统管理硬件资源、CANN 作为中间层连接上层框架与底层 NPU 的具体体现。

下方代码在 NPU 上创建 2048×2048 的随机矩阵并执行矩阵乘法，然后与 CPU 计算耗时对比，展示 Host-Device 异构协作的加速效果。

**预期结果**：在昇腾环境中，会输出 PyTorch 和 torch_npu 版本、NPU 设备信息，NPU 矩阵乘法耗时通常远小于 CPU，加速比可达数十倍。在非昇腾环境中，会提示 `torch 或 torch_npu 未安装`，属正常现象——说明当前环境没有昇腾 NPU 硬件和 CANN 软件栈。

In [ ]:
import time

try:
    import torch
    import torch_npu

    print(f"PyTorch 版本: {torch.__version__}")
    print(f"NPU 可用: {torch.npu.is_available()}")
    print(f"NPU 卡数: {torch.npu.device_count()}")
    print(f"NPU 名称: {torch.npu.get_device_name(0)}")

    N = 2048
    # 方式一：直接在 NPU 上创建数据（推荐）
    A = torch.randn(N, N, device='npu:0')
    B = torch.randn(N, N, device='npu:0')
    print(f"\n[Device] 矩阵创建在 NPU 上, A.device={A.device}")

    # NPU 执行矩阵乘法
    torch.npu.synchronize()
    t0 = time.time()
    C = torch.mm(A, B)
    torch.npu.synchronize()
    t_npu = time.time() - t0
    print(f"[Device] NPU 矩阵乘法完成, 耗时: {t_npu*1000:.2f} ms, 结果形状: {C.shape}")

    # Device -> Host 搬运结果
    C_cpu = C.cpu()
    print(f"[Device -> Host] 结果搬回 CPU, C.device={C_cpu.device}")

    # 对比 CPU 计算耗时
    A_cpu = torch.randn(N, N)
    B_cpu = torch.randn(N, N)
    t0 = time.time()
    C_ref = torch.mm(A_cpu, B_cpu)
    t_cpu = time.time() - t0
    print(f"\n[对比] CPU 矩阵乘法耗时: {t_cpu*1000:.2f} ms")
    print(f"[对比] NPU 加速比: {t_cpu/t_npu:.1f}x")

except ImportError:
    print("[!] torch 或 torch_npu 未安装")
    print("    在昇腾 CANN 环境中运行可体验完整的 Host-Device 异构协作")
except Exception as e:
    print(f"[!] 运行异常: {e}")

---

## 小结

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">概念</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">嵌入式操作系统</td>
<td style="text-align: left;">运行在嵌入式设备上的 OS，资源受限、硬件耦合、常需实时性</td>
</tr>
<tr>
<td style="text-align: left;">昇腾四层架构</td>
<td style="text-align: left;">操作系统层 → CANN 层 → 框架与模型层 → 应用与工具层</td>
</tr>
<tr>
<td style="text-align: left;">双轨制 OS</td>
<td style="text-align: left;">Ubuntu（创新沙盒）+ openEuler（生产环境）</td>
</tr>
<tr>
<td style="text-align: left;">启动流程</td>
<td style="text-align: left;">BootLoader → Linux Kernel → RootFS → User Apps</td>
</tr>
<tr>
<td style="text-align: left;">定制内核</td>
<td style="text-align: left;">基于 LTS 深度定制，集成昇腾专用驱动和加速模块</td>
</tr>
<tr>
<td style="text-align: left;">三种驱动方案</td>
<td style="text-align: left;">传统内核驱动 / 用户空间驱动 / 寄存器级驱动</td>
</tr>
<tr>
<td style="text-align: left;">wiringOP</td>
<td style="text-align: left;">香橙派 GPIO/SPI/I2C 控制库，Arduino 风格 API</td>
</tr>
<tr>
<td style="text-align: left;">NPU-SMI</td>
<td style="text-align: left;">昇腾 NPU 管理工具，监控温度/功耗/利用率/内存</td>
</tr>
<tr>
<td style="text-align: left;">系统构建</td>
<td style="text-align: left;">Buildroot（轻量）/ Yocto（工业级）/ 手动构建（极致定制）</td>
</tr>
</table>

**小结表解读**：本表是对全章核心知识点的浓缩回顾。"嵌入式操作系统"强调与桌面 OS 的三大差异（资源、实时性、硬件耦合）；"昇腾四层架构"概括了从硬件到应用的完整软件栈层次；"双轨制 OS"体现了昇腾兼顾入门与生产的生态策略；"启动流程"描述了嵌入式 Linux 从上电到运行的四个阶段；"定制内核"说明了昇腾为何需要深度定制 Linux 内核；"三种驱动方案"对比了不同复杂度和性能的驱动开发路径；"wiringOP"是香橙派上进行外设控制的用户态库；"NPU-SMI"是昇腾硬件的监控管理工具；"系统构建"列出了从轻量到极致定制的三种嵌入式 Linux 构建方法。建议学生对照此表逐一回顾各节内容，确保每个概念都已理解。

---

## 课后练习

请根据本节课程学习内容完成以下题目进行自测，在每题下方的代码框中输入选项字母后运行。


**第1题**（单选题）嵌入式操作系统与桌面操作系统相比，最显著的特点是？

- A. 功能更强大
- B. 资源受限且与特定硬件紧密耦合
- C. 用户界面更友好
- D. 不需要文件系统


In [ ]:
q1 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第{1}题答案已记录：{q1}' if q1 else '⚠️ 请填入答案并运行本单元格')

**第2题**（单选题）昇腾软件体系架构共有几层？

- A. 2层
- B. 3层
- C. 4层
- D. 5层


In [ ]:
q2 = ''
print(f'第{2}题答案已记录：{q2}' if q2 else '⚠️ 请填入答案并运行本单元格')

**第3题**（单选题）CANN 在昇腾软件体系中的定位是？

- A. 操作系统层
- B. 异构计算架构层（连接上层生态与底层硬件的桥梁）
- C. 框架与模型层
- D. 应用与工具层


In [ ]:
q3 = ''
print(f'第{3}题答案已记录：{q3}' if q3 else '⚠️ 请填入答案并运行本单元格')

**第4题**（单选题）昇腾双轨制操作系统中，openEuler 的定位是？

- A. 大众化入门·创新沙盒
- B. 企业级平台·生产环境
- C. 嵌入式实时操作系统
- D. 桌面操作系统


In [ ]:
q4 = ''
print(f'第{4}题答案已记录：{q4}' if q4 else '⚠️ 请填入答案并运行本单元格')

**第5题**（单选题）嵌入式 Linux 系统启动流程的正确顺序是？

- A. Linux Kernel → BootLoader → RootFS → User Apps
- B. BootLoader → RootFS → Linux Kernel → User Apps
- C. BootLoader → Linux Kernel → RootFS → User Apps
- D. RootFS → BootLoader → Linux Kernel → User Apps


In [ ]:
q5 = ''
print(f'第{5}题答案已记录：{q5}' if q5 else '⚠️ 请填入答案并运行本单元格')

**第6题**（单选题）嵌入式 Linux 启动时，内核启动的第一个用户空间进程是？

- A. /bin/bash
- B. /sbin/init
- C. /usr/bin/python
- D. /boot/grub


In [ ]:
q6 = ''
print(f'第{6}题答案已记录：{q6}' if q6 else '⚠️ 请填入答案并运行本单元格')

**第7题**（单选题）昇腾平台为什么需要定制 Linux 内核？

- A. 为了节省磁盘空间
- B. 昇腾专用 AI 硬件架构必须深度定制内核才能充分发挥计算潜能
- C. 为了支持图形界面
- D. 为了兼容 Windows 应用


In [ ]:
q7 = ''
print(f'第{7}题答案已记录：{q7}' if q7 else '⚠️ 请填入答案并运行本单元格')

**第8题**（单选题）Linux 驱动开发三种方案中，开发最简单但性能受限的是？

- A. 传统内核驱动
- B. 纯用户空间驱动
- C. 寄存器级驱动
- D. 三者难度相同


In [ ]:
q8 = ''
print(f'第{8}题答案已记录：{q8}' if q8 else '⚠️ 请填入答案并运行本单元格')

**第9题**（单选题）NPU-SMI 工具的功能定位类似于 NVIDIA 的哪个工具？

- A. nvcc
- B. nvidia-smi
- C. cudnn
- D. tensorrt


In [ ]:
q9 = ''
print(f'第{9}题答案已记录：{q9}' if q9 else '⚠️ 请填入答案并运行本单元格')

**第10题**（单选题）嵌入式 Linux 系统构建方法中，适合工业级应用（如汽车电子）的是？

- A. Buildroot
- B. Yocto
- C. 手动构建
- D. apt-get install


In [ ]:
q10 = ''
print(f'第{10}题答案已记录：{q10}' if q10 else '⚠️ 请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path

for candidate in (
    Path.cwd() / 'answer',
    Path.cwd() / 'quick_start' / 'cann_basics' / 'answer',
):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_03 import grade
grade(globals())

## 参考资料

- [昇腾 CANN 文档](https://www.hiascend.com/document)
- [openEuler 社区](https://www.openeuler.org/)
- [wiringOP 库](https://github.com/orangepi-xunlong/wiringOP)
- [U-Boot 官方文档](https://docs.u-boot.org/)
- [Yocto Project](https://www.yoctoproject.org/)
- [Buildroot](https://buildroot.org/)
